# College Scorecard Analysis from DOE 

Hypothesis 1 (College Scorecard): Students who attend more racially and demographically diverse universities will demonstrate higher median earnings after graduation, reflecting the broad economic and social benefits of diverse educational environments.

- Initial datasets are too large, so needed to cut down the variables included in the analysis
- Merged initial college scorecard dataset with additional variables that tell me additional information about the university which I can include as controls (e.g. whether the college is an HBCU, etc.)


### Setup

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')
from IPython.display import IFrame

# Data Cleaning Process

1. Read in the datasets and create a dictionary for the key variables that I want to keep
2. Find the variables that start with certain headings to isolate similar variables within different groups (e.g. variables starting with ugds_ are all variables that give the percent of each racial group at a university).

In [2]:
#data cleaning process-- especially making smaller datasets that are more manageable and only include relevant variables

# dat1 = pd.read_csv('../data/Glynn_data/MERGED2021_22_PP.csv')
# dat2 = pd.read_csv('../data/Glynn_data/MERGED2022_23_PP.csv')
# #making a general dictionary of variables that I think will be important for the analysis
# name_dict = { #general variables for id
#     'UNITID': 'univ_id',
#     'INSTNM': 'univ_name',
# #    'CIPCODE': 'field_code', #field being what students study, but not in correct datasets
# #    'CIPDESC': 'fielf_desc',
# #    'CREDDESC': 'degree_type',
# #    'EARN_MDN_4YR': 'median_earnings',
#     'STABBR': 'state', 
#     'ZIP': 'zip_code',
#     'HBCU': 'hbcu',
#     'PBI': 'pbi', #predominantly black institution (40%+ of student body is Black)
#     'AANAPII': 'aanapii', 
#     'HSI': 'hsi',
#     'TRIBAL': 'tribal', 
#     'PCTPELL': 'percent_pell',
#     'C150_4': 'completion_rate_fry'
# }

In [3]:
# #finding racial demographics (all coded as starting with UGDS_)
# race_cols = {}
# for col in dat1.columns:
#     if col.startswith('ugds_'):
#         race_cols[col] = col.lower()

# race_cols

In [4]:
# #finding median earnings data (all coded as starting with MD_EARN)
# earn_cols = {}
# for col in dat1.columns:
#     if col.startswith('MD_EARN'): 
#         earn_cols[col] = col.lower()

In [5]:
# #finding family income distribution (all coded as starting with INC_)
# income_cols = {}
# for col in dat1.columns:
#     if col.startswith('INC_'): 
#         income_cols[col] = col.lower()
# income_cols

In [6]:
# #making sure that the new parts of the name dictionary are reflected
# name_dict.update(race_cols)
# name_dict.update(earn_cols)
# name_dict.update(income_cols)
# len(name_dict) #88 unique variables

In [7]:
# data_21 = (dat1[list(name_dict.keys())]
#         .rename(columns=name_dict))
# data_21.columns #a manageable amount of columns, encompassing all the variables I need
# # new csv file
# data_21['year'] = 2021

# len(data_21.columns)

## Saving the new cleaned datasets for further use and analysis 

After the initial cleaning steps, I narrowed the datasets only to the relevant variables that I identified in the dictionary and earlier analysis. After finding those variables, I give them new names based on the key value pairs. In an earlier iteration of this notebook, I also made a year column that became irrelevant later. 

In [8]:
# data_21.to_csv('data/Glynn_data/clean_21_data.csv', index=False) #only 1.2 mb. 

In [9]:
# #repeating the process with the 2022 data, assuming that the name_dict is the same for 2022, please god
# data_22 = (dat2[list(name_dict.keys())]
#           .rename(columns = name_dict))
# data_22['year'] = 2022
# len(data_22.columns) #checking for a match between the types of columns in both datset

In [10]:
# data_22.to_csv('data/Glynn_data/clean_22_data.csv', index=False) #also 1.2 mb. 

# Second round of filtration

Already finished filtering through and saving the larger dataset so github doesn't crash out. Need to reload the dat2 dataset which has already been cleaned and narrowed down. 

Step 1: clean the most-recent-cohorts file, which is the file from the DOE that has the pct Black, completion rates for people from their hometown, and other variables that can be merged to the university by univ_id. 

Step 2: only keep track of columns that are relvant to the analysis from the coh dataset. 

Step 3: merge with the dat2 2022 data according to the unviersity ID

In [11]:
#in case starting from these lines of code: comment out later or just start from here unless you need to filter out other datasets
#dat1 = pd.read_csv('../data/Glynn_data/clean_21_data.csv')
dat2 = pd.read_csv('../data/Glynn_data/clean_22_data.csv')
coh = pd.read_csv('../data/Glynn_data/Most-Recent-Cohorts-Institution.csv')

In [12]:
dat2 = dat2.dropna(axis=1, how='all')
dat2.head()

,univ_id,univ_name,state,zip_code,percent_pell,completion_rate_fry,ugds_white,ugds_black,ugds_hisp,ugds_asian,ugds_aian,ugds_nhpi,ugds_2mor,ugds_nra,ugds_unkn,ugds_men,ugds_women,md_earn_wne_4yr,year
0,100654,Alabama A & M University,AL,35762,0.6536,0.2678,0.0198,0.8955,0.0110,0.0019,0.0025,0.0015,0.0127,0.0115,0.0435,0.4055,0.5945,54018,2022
1,100663,University of Alabama at Birmingham,AL,35294-0110,0.3308,0.6442,0.5130,0.2528,0.0711,0.0819,0.0016,0.0005,0.0491,0.0237,0.0064,0.3752,0.6248,64969,2022
2,100690,Amridge University,AL,36117-3553,0.7769,0.5000,0.2851,0.6623,0.0307,0.0000,0.0044,0.0044,0.0000,0.0000,0.0132,0.3640,0.6360,50075,2022
3,100706,University of Alabama in Huntsville,AL,35899,0.2173,0.6295,0.7102,0.0873,0.0666,0.0389,0.0087,0.0016,0.0465,0.0149,0.0252,0.5981,0.4019,81045,2022
4,100724,Alabama State University,AL,36104-0271,0.6976,0.2773,0.0155,0.9251,0.0121,0.0015,0.0021,0.0009,0.0118,0.0221,0.0088,0.3595,0.6405,45998,2022


In [13]:
name_dict = { #general variables for id. Some commented variables already included from earlier iterations of the 
        #data cleaning process
    'UNITID': 'univ_id',
#    'INSTNM': 'univ_name',
#    'CIPCODE': 'field_code', #field being what students study, but not in correct datasets
#    'CIPDESC': 'fielf_desc',
#    'CREDDESC': 'degree_type',
#    'EARN_MDN_4YR': 'median_earnings',
#    'STABBR': 'state', 
#    'ZIP': 'zip_code',
    'HBCU': 'hbcu',
    'PBI': 'pbi', #predominantly black institution (40%+ of student body is Black)
    'AANAPII': 'aanapii', 
    'HSI': 'hsi',
    'TRIBAL': 'tribal', 
#    'PCTPELL': 'percent_pell',
#    'C150_4': 'completion_rate_fry', #completion rate for four year institutions (if NA does this mean not a 4yr?)
    'LONGITUDE': 'long',
    'MENONLY': 'all_male', 
    'WOMENONLY': 'all_female', 
    'ADM_RATE': 'admit_rate', #measure of exclusivity
    'LATITUDE': 'lat', 
    'PCT_BA': 'pct_w_ba_hometown', #Percent of the population from students' zip codes with a bachelor's degree over the age 25, via Census data
    'PREDDEG': 'degree_type' #"Predominant undergraduate degree awarded: 0 Not classified, 1 Predominantly certificate-degree granting, 
                    #2 Predominantly associate's-degree granting, 3 Predominantly bachelor's-degree granting,  4 Entirely graduate-degree granting"
}


#dict 2, specifically for the undergrad 

In [14]:
len(name_dict) #13 unique variables

13

In [15]:
#repeating the process with the 2022 data, assuming that the name_dict is the same for 2022, please god
coh_filtered = (coh[list(name_dict.keys())]
          .rename(columns = name_dict))
#data_22['year'] = 2022
len(coh_filtered.columns) #checking for a match between the types of columns in both datset

13

In [16]:
coh_filtered.head()

,univ_id,hbcu,pbi,aanapii,hsi,tribal,long,all_male,all_female,admit_rate,lat,pct_w_ba_hometown,degree_type
0,100654,1.0,0.0,0.0,0.0,0.0,-86.568502,0.0,0.0,0.5795,34.783368,13,3
1,100663,0.0,0.0,0.0,0.0,0.0,-86.799345,0.0,0.0,0.8818,33.505697,15.9300003051757,3
2,100690,0.0,0.0,0.0,0.0,0.0,-86.174010,0.0,0.0,NaN,32.362609,13.2299995422363,3
3,100706,0.0,0.0,0.0,0.0,0.0,-86.640449,0.0,0.0,0.6857,34.724557,17.6700000762939,3
4,100724,1.0,0.0,0.0,0.0,0.0,-86.295677,0.0,0.0,0.9755,32.364317,11.8100004196167,3


## Merging

After reloading and filtering down the datasets, merge the datasets with the other university information. 

Convert this new dataset to a CSV and save it to the data file

In [17]:
#making a combined dataframe to work with
combined_dat = pd.merge(dat2, coh_filtered, on = 'univ_id')
#Example: pd.merge(df1, df2, on='id', how='left'). 

In [18]:
combined_dat.info()

<class 'pandas.DataFrame'>
RangeIndex: 6159 entries, 0 to 6158
Data columns (total 31 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   univ_id              6159 non-null   int64  
 1   univ_name            6159 non-null   str    
 2   state                6159 non-null   str    
 3   zip_code             6159 non-null   str    
 4   percent_pell         5442 non-null   float64
 5   completion_rate_fry  2219 non-null   float64
 6   ugds_white           5483 non-null   float64
 7   ugds_black           5483 non-null   float64
 8   ugds_hisp            5483 non-null   float64
 9   ugds_asian           5483 non-null   float64
 10  ugds_aian            5483 non-null   float64
 11  ugds_nhpi            5483 non-null   float64
 12  ugds_2mor            5483 non-null   float64
 13  ugds_nra             5483 non-null   float64
 14  ugds_unkn            5483 non-null   float64
 15  ugds_men             5483 non-null   float64
 16 

In [19]:
#convert to a csv
combined_dat.to_csv('../data/Glynn_data/combined_data.csv', index = False)